<a href="https://colab.research.google.com/github/arman-hossain45/Deep_Learning_Lab/blob/main/practice_text_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TExt classification

In [24]:

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, GRU, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings('ignore')


In [25]:
df= pd.read_csv('df_file.csv')

In [26]:
df.head(6)

,Text,Label
0,Budget to set scene for election\n \n Gordon B...,0
1,Army chiefs in regiments decision\n \n Militar...,0
2,Howard denies split over ID cards\n \n Michael...,0
3,Observers to monitor UK election\n \n Minister...,0
4,Kilroy names election seat target\n \n Ex-chat...,0
5,Donor attacks Blair-Brown 'feud'\n \n The repo...,0


In [27]:

# # ==========================================================



from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

Configuration

In [28]:
df.columns

Index(['Text', 'Label'], dtype='object')

In [29]:
df=df[['Text', 'Label']]

In [30]:
df.isnull().sum()

,0
Text,0
Label,0


In [31]:
df=df.dropna()

In [32]:
label_encoder = LabelEncoder()
df['Label'] = label_encoder.fit_transform(df['Label'])

In [33]:
df['Label']

,Label
0,0
1,0
2,0
3,0
4,0
...,...
2220,4
2221,4
2222,4
2223,4


In [34]:
label_encoder.classes_

array([0, 1, 2, 3, 4])

In [35]:
text=df['Text'].astype(str)
labels=df['Label']

In [36]:
len(df)

2225

In [37]:
x_train,x_test,y_train,y_test=train_test_split(text,labels,test_size=0.2,random_state=42,stratify=labels)

In [38]:
max_features=10000
max_len=500

In [42]:
x_train_text, x_test_text, _, _ = train_test_split(text, labels, test_size=0.2, random_state=42, stratify=labels)

# Tokenization
tokenizer = Tokenizer(num_words=max_features, oov_token='')
tokenizer.fit_on_texts(x_train_text)

x_train = tokenizer.texts_to_sequences(x_train_text)
x_test = tokenizer.texts_to_sequences(x_test_text)

# Padding
x_train = pad_sequences(x_train, maxlen=max_len, padding='post', truncating='post')
x_test = pad_sequences(x_test, maxlen=max_len, padding='post', truncating='post')

In [43]:
x_train

array([[ 246,  822,  234, ...,    0,    0,    0],
       [4076, 7219, 2514, ...,    0,    0,    0],
       [  15, 3938,  131, ...,    2,  608,   18],
       ...,
       [   2,  254,    7, ...,  575,   27,    6],
       [2402,  764, 6522, ...,    0,    0,    0],
       [ 476,    1,  241, ...,    0,    0,    0]], dtype=int32)

In [44]:
x_test

array([[1864,  577,  206, ...,    0,    0,    0],
       [ 194,  159,    1, ...,   52,  769, 1289],
       [ 848,  849,   11, ..., 4329,    3,    2],
       ...,
       [  49, 2476,  367, ...,    0,    0,    0],
       [ 485, 1724, 9506, ...,    0,    0,    0],
       [2061,   70, 1755, ...,    0,    0,    0]], dtype=int32)

In [57]:
epochs=10
batch_size=64

In [58]:
#create rnn model
model = Sequential(
    [
        Embedding(input_dim=max_features,output_dim=128,input_length=max_len),
        SimpleRNN(64,return_sequences=False),
        Dropout(0.5),
        Dense(64,activation='relu'),
        Dropout(0.5),
        Dense(5,activation='softmax')
    ]
)


model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

#early stopping
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

history = model.fit(
        x_train, y_train,
        validation_data=(x_test, y_test),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[early_stopping],
        verbose=1
    )

Epoch 1/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 5s 130ms/step - accuracy: 0.2309 - loss: 1.6192 - val_accuracy: 0.2472 - val_loss: 1.5897
Epoch 2/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 6s 149ms/step - accuracy: 0.2556 - loss: 1.6018 - val_accuracy: 0.2584 - val_loss: 1.5801
Epoch 3/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 123ms/step - accuracy: 0.3090 - loss: 1.5423 - val_accuracy: 0.2989 - val_loss: 1.5654
Epoch 4/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 4s 127ms/step - accuracy: 0.3371 - loss: 1.5110 - val_accuracy: 0.2764 - val_loss: 1.5553
Epoch 5/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 6s 148ms/step - accuracy: 0.3388 - loss: 1.4600 - val_accuracy: 0.2742 - val_loss: 1.5421
Epoch 6/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 118ms/step - accuracy: 0.3702 - loss: 1.4208 - val_accuracy: 0.2652 - val_loss: 1.5310
Epoch 7/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 122ms/step - accuracy: 0.3882 - loss: 1.3737 - val_accuracy: 0.2674 - val_loss: 1.5504
Epoch 8/10
28/28 ━━━━━━━━━━━━━━━━━━━━ 4s 144ms/step - accuracy: 0.4017 - loss: 1.3500 - val_accuracy: 0.

In [60]:
# for binary classification then apply this code in details
 # Predict
    # y_pred_proba = model.predict(X_test)
    # y_pred = (y_pred_proba > 0.5).astype(int)



# for multiclass classification then apply this code
y_pred_proba= model.predict(x_test)
y_pred =np.argmax(y_pred_proba,axis=1)
# calculate the metrics
accuracy =accuracy_score(y_test,y_pred)
report = classification_report(y_test,y_pred)
cm=confusion_matrix(y_test,y_pred)

14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step


In [61]:
accuracy

0.2651685393258427

In [62]:
report

'              precision    recall  f1-score   support\n\n           0       0.39      0.23      0.29        84\n           1       0.24      0.75      0.36       102\n           2       0.35      0.23      0.27        80\n           3       0.00      0.00      0.00        77\n           4       0.17      0.05      0.08       102\n\n    accuracy                           0.27       445\n   macro avg       0.23      0.25      0.20       445\nweighted avg       0.23      0.27      0.20       445\n'

In [63]:
cm

array([[19, 42, 14,  0,  9],
       [ 9, 76,  9,  0,  8],
       [16, 43, 18,  0,  3],
       [ 1, 67,  5,  0,  4],
       [ 4, 87,  6,  0,  5]])